# 🔁 Day 05a: RNN & LSTM — Sequence Modeling

---

## 🎯 What You'll Master Today
- RNN cell mechanics & vanishing gradient problem
- LSTM gates (forget, input, output)
- GRU as simplified LSTM
- Build sequence models from memory

**Goal:** Implement LSTM cell from scratch. Know the gates cold.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  RNN vs LSTM vs GRU                                             ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  RNN:   h_t = tanh(W_hh · h_{t-1} + W_xh · x_t + b)           ║
║         Problem: vanishing gradient (long sequences fail)       ║
║                                                                   ║
║  LSTM:  Adds cell state C_t (highway for gradients)             ║
║         3 gates:                                                 ║
║           forget gate  f = σ(W_f · [h,x] + b_f)   → what to   ║
║                                                       forget   ║
║           input gate   i = σ(W_i · [h,x] + b_i)   → what to   ║
║                                                       store    ║
║           output gate  o = σ(W_o · [h,x] + b_o)   → what to   ║
║                                                       output   ║
║         C_t = f * C_{t-1} + i * tanh(W_c · [h,x] + b_c)      ║
║         h_t = o * tanh(C_t)                                     ║
║                                                                   ║
║  GRU:   2 gates (update, reset). Simpler, often comparable.     ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# ============================================================
# 1. Vanilla RNN Cell — From Scratch
# ============================================================

class RNNCell:
    """Vanilla RNN: h_t = tanh(W_hh @ h + W_xh @ x + b)"""
    def __init__(self, input_size, hidden_size):
        scale = 1 / np.sqrt(hidden_size)
        self.W_xh = np.random.randn(input_size, hidden_size) * scale
        self.W_hh = np.random.randn(hidden_size, hidden_size) * scale
        self.b = np.zeros(hidden_size)
    
    def forward(self, x, h_prev):
        """x: (input_size,), h_prev: (hidden_size,)"""
        h_next = np.tanh(x @ self.W_xh + h_prev @ self.W_hh + self.b)
        return h_next

# Demo
rnn_cell = RNNCell(input_size=10, hidden_size=20)
h = np.zeros(20)  # initial hidden state

# Process a sequence of 5 time steps
for t in range(5):
    x_t = np.random.randn(10)
    h = rnn_cell.forward(x_t, h)
    print(f"t={t}: h norm = {np.linalg.norm(h):.4f}")

print("\n💡 Simple RNN works but gradients vanish/explode for long sequences")

In [ ]:
# ============================================================
# 2. LSTM Cell — From Scratch (The Key Interview Drill)
# ============================================================

class LSTMCell:
    """
    LSTM with forget, input, output gates + cell state.
    
    f_t = sigmoid(W_f @ [h, x] + b_f)    # forget gate
    i_t = sigmoid(W_i @ [h, x] + b_i)    # input gate
    c_candidate = tanh(W_c @ [h, x] + b_c)  # candidate
    o_t = sigmoid(W_o @ [h, x] + b_o)    # output gate
    
    C_t = f_t * C_{t-1} + i_t * c_candidate
    h_t = o_t * tanh(C_t)
    """
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        combined = input_size + hidden_size
        scale = 1 / np.sqrt(hidden_size)
        
        # One weight matrix per gate
        self.W_f = np.random.randn(combined, hidden_size) * scale
        self.b_f = np.ones(hidden_size)  # init forget bias to 1 (remember by default)
        
        self.W_i = np.random.randn(combined, hidden_size) * scale
        self.b_i = np.zeros(hidden_size)
        
        self.W_c = np.random.randn(combined, hidden_size) * scale
        self.b_c = np.zeros(hidden_size)
        
        self.W_o = np.random.randn(combined, hidden_size) * scale
        self.b_o = np.zeros(hidden_size)
    
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    def forward(self, x, h_prev, c_prev):
        """x: (input_size,), h_prev: (hidden_size,), c_prev: (hidden_size,)"""
        # Concatenate input and previous hidden state
        combined = np.concatenate([h_prev, x])
        
        # Gates
        f = self.sigmoid(combined @ self.W_f + self.b_f)  # forget
        i = self.sigmoid(combined @ self.W_i + self.b_i)  # input
        c_cand = np.tanh(combined @ self.W_c + self.b_c)  # candidate
        o = self.sigmoid(combined @ self.W_o + self.b_o)  # output
        
        # Cell state update
        c_next = f * c_prev + i * c_cand
        
        # Hidden state
        h_next = o * np.tanh(c_next)
        
        return h_next, c_next, {'f': f, 'i': i, 'o': o}

# Demo LSTM
lstm_cell = LSTMCell(input_size=10, hidden_size=20)
h = np.zeros(20)
c = np.zeros(20)

for t in range(5):
    x_t = np.random.randn(10)
    h, c, gates = lstm_cell.forward(x_t, h, c)
    print(f"t={t}: h_norm={np.linalg.norm(h):.4f}, "
          f"c_norm={np.linalg.norm(c):.4f}, "
          f"forget_mean={gates['f'].mean():.3f}")

print("\n💡 Cell state C acts as a highway — gradients flow through f_t multiplications")
print("   Forget bias initialized to 1 → default: remember everything")

In [ ]:
# ============================================================
# 3. PyTorch LSTM — How to Use It
# ============================================================

# nn.LSTM input: (seq_len, batch, input_size)
# or (batch, seq_len, input_size) with batch_first=True

lstm = nn.LSTM(
    input_size=10,    # feature dim at each time step
    hidden_size=20,   # hidden state dim
    num_layers=2,     # stacked LSTM layers
    batch_first=True, # input shape: (batch, seq_len, features)
    dropout=0.2,      # between LSTM layers (not applied to last layer)
    bidirectional=False,
)

# Input: batch=4, seq_len=15, features=10
x = torch.randn(4, 15, 10)

# Forward pass
output, (h_n, c_n) = lstm(x)

print("LSTM input/output shapes:")
print(f"  Input:  {x.shape}          # (batch, seq_len, input_size)")
print(f"  Output: {output.shape}   # (batch, seq_len, hidden_size)")
print(f"  h_n:    {h_n.shape}       # (num_layers, batch, hidden_size)")
print(f"  c_n:    {c_n.shape}       # (num_layers, batch, hidden_size)")

print("\n💡 output = hidden states at ALL time steps")
print("   h_n    = hidden state at LAST time step (per layer)")
print("   For classification: use h_n[-1] or output[:, -1, :]")

# Verify: last output == last layer's h_n
print(f"\noutput[:,-1,:] == h_n[-1]: {torch.allclose(output[:, -1, :], h_n[-1])}")

In [ ]:
# ============================================================
# 4. Sequence Classifier with LSTM
# ============================================================

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.3 if num_layers > 1 else 0,
        )
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # x: (batch, seq_len) of token indices
        embedded = self.embedding(x)       # (batch, seq_len, embed_dim)
        output, (h_n, c_n) = self.lstm(embedded)
        # Use last hidden state of last layer
        last_hidden = h_n[-1]              # (batch, hidden_dim)
        out = self.fc(self.dropout(last_hidden))
        return out

# Demo: sentiment classifier
model = LSTMClassifier(
    vocab_size=5000,
    embed_dim=128,
    hidden_dim=256,
    num_classes=2,   # positive/negative
    num_layers=2,
)

# Fake batch of token sequences
batch = torch.randint(0, 5000, (8, 50))  # 8 sentences, 50 tokens each
logits = model(batch)
print(f"Input: {batch.shape} → Output: {logits.shape}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ============================================================
# 5. Bidirectional & GRU
# ============================================================

# ----- Bidirectional LSTM -----
bi_lstm = nn.LSTM(10, 20, batch_first=True, bidirectional=True)
x = torch.randn(4, 15, 10)
out, (h_n, c_n) = bi_lstm(x)

print("Bidirectional LSTM:")
print(f"  Output: {out.shape}   # hidden_size × 2 = 40")
print(f"  h_n:    {h_n.shape}       # num_layers × 2 directions")

# For classification with BiLSTM: concat forward and backward final states
h_forward = h_n[0]   # forward direction, last layer
h_backward = h_n[1]  # backward direction, last layer
h_cat = torch.cat([h_forward, h_backward], dim=1)
print(f"  Concat h: {h_cat.shape}  # use this for classification FC")

# ----- GRU (simpler alternative) -----
print("\nGRU:")
gru = nn.GRU(10, 20, batch_first=True)
out, h_n = gru(x)  # GRU has NO cell state, just h_n
print(f"  Output: {out.shape}")
print(f"  h_n:    {h_n.shape}")

print("""
╔══════════════════════════════════════════════╗
║  RNN Variant Comparison                      ║
╠══════════════════════════════════════════════╣
║  Vanilla RNN │ Fast, simple, vanishing grad  ║
║  LSTM        │ Cell state fixes vanishing     ║
║              │ 3 gates, more params           ║
║  GRU         │ 2 gates, fewer params          ║
║              │ Often matches LSTM performance ║
║  BiLSTM      │ Sees future context too        ║
║              │ 2× params, best for NLP tasks  ║
╚══════════════════════════════════════════════╝
""")

---

## 🗺️ RNN/LSTM Pattern Map

```
╔═══════════════════════════════════════════════════════════════════╗
║  SEQUENCE MODELING ESSENTIALS                                    ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  RNN input: (batch, seq_len, features)  with batch_first=True  ║
║  Output: hidden states at ALL steps + final (h_n, c_n)         ║
║                                                                   ║
║  LSTM gates (memorize these!):                                   ║
║    forget: f = σ(W_f·[h,x] + b_f)  → erase old info           ║
║    input:  i = σ(W_i·[h,x] + b_i)  → write new info           ║
║    output: o = σ(W_o·[h,x] + b_o)  → read cell state          ║
║    C_t = f·C_{t-1} + i·tanh(W_c·[h,x] + b_c)                 ║
║    h_t = o·tanh(C_t)                                            ║
║                                                                   ║
║  Classification: use h_n[-1] (last hidden of last layer)       ║
║  Seq2Seq: use all outputs                                       ║
║                                                                   ║
║  Modern NLP: Transformers mostly replaced LSTM                  ║
║  But LSTM still used in: time series, speech, edge devices      ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | Why does vanilla RNN fail on long sequences? | Vanishing gradient — repeated multiplication of small values through time. Gradient → 0 |
| 2 | How does LSTM solve vanishing gradient? | Cell state C_t acts as gradient highway. Forget gate uses addition (not multiplication), preserving gradient flow |
| 3 | What are the 3 LSTM gates? | Forget (what to erase), Input (what to store), Output (what to expose). All are sigmoid → [0,1] |
| 4 | LSTM vs GRU? | GRU: 2 gates (update, reset), fewer params, often similar performance. LSTM: 3 gates + cell state, more expressive, standard choice |
| 5 | When is BiLSTM useful? | When you need future context (NLP classification, NER). Not for autoregressive tasks (text generation) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • LSTM = forget + input + output gates + cell state    │
## │  • Cell state = gradient highway (solves vanishing)     │
## │  • Forget bias → 1 (remember by default)                │
## │  • output[:, -1, :] == h_n[-1] (last hidden state)     │
## │  • BiLSTM: 2× hidden dim, concat forward+backward      │
## │  • GRU: simpler, 2 gates, often comparable              │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## 🏁 PyTorch Chapter Complete!
## ⏭️ Next Chapter: **TensorFlow/Keras** — the other framework